<a href="https://colab.research.google.com/github/somustafa/movierecc/blob/main/movierecc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [51]:
%%writefile streamlit_app.py


Overwriting streamlit_app.py


In [52]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors
import re
from sklearn.neighbors import NearestNeighbors

In [53]:
ratings = pd.read_csv('ratings.csv')
movies = pd.read_csv('movies.csv')

In [54]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [55]:
ratings.head()


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [56]:
ratings.info()
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB


In [57]:
ratings.isnull().sum()
movies.isnull().sum()
# oyrenirik ki null value miz yoxdur yeni ki datasetimiz temizdir missing value yoxdur

,0
movieId,0
title,0
genres,0


In [58]:
print(ratings.shape) #raw, columns qayatrir
print(movies.shape) # nece setir var ve nece column var onu oyrenmis olduq


(100836, 4)
(9742, 3)


In [59]:
ratings['userId'].nunique() #unikal userler

610

In [60]:
ratings['movieId'].nunique() #unikal filmler

9724

In [61]:
#bura qeder her bir seyi datasetden xeberimiz olsun deye eledik.
ratings['rating'].value_counts().sort_index() # her ratingden data setde ne qeder var baxiriq

,count
rating,
0.5,1370
1.0,2811
1.5,1791
2.0,7551
2.5,5550
3.0,20047
3.5,13136
4.0,26818
4.5,8551


In [62]:
print("Ratings shape:", ratings.shape)
print("Movies shape:", movies.shape)
print("Unique users:", ratings['userId'].nunique())
print("Unique movies:", ratings['movieId'].nunique())
print(ratings['rating'].value_counts().sort_index())


Ratings shape: (100836, 4)
Movies shape: (9742, 3)
Unique users: 610
Unique movies: 9724
rating
0.5     1370
1.0     2811
1.5     1791
2.0     7551
2.5     5550
3.0    20047
3.5    13136
4.0    26818
4.5     8551
5.0    13211
Name: count, dtype: int64


In [63]:
user_item_matrix = ratings.pivot(index='userId', columns='movieId', values='rating')
user_item_matrix = user_item_matrix.fillna(0)  # NaN → 0 (rating verilməyib)
user_item_matrix.head()

movieId,1,2,3,4,5,6,7,8,9,10,...,193565,193567,193571,193573,193579,193581,193583,193585,193587,193609
userId,,,,,,,,,,,,,,,,,,,,,
1,4.0,0.0,4.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [64]:
movies['genres'] = movies['genres'].apply(lambda x: x.split('|'))


In [65]:
print(movies.head())


   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                              genres  
0  [Adventure, Animation, Children, Comedy, Fantasy]  
1                     [Adventure, Children, Fantasy]  
2                                  [Comedy, Romance]  
3                           [Comedy, Drama, Romance]  
4                                           [Comedy]  


In [66]:
#title prosessing
import re

def extract_year(title):
    match = re.search(r'\((\d{4})\)', title)
    return int(match.group(1)) if match else None

movies['year'] = movies['title'].apply(extract_year)
movies['clean_title'] = movies['title'].apply(lambda x: re.sub(r'\(\d{4}\)', '', x).strip())


In [67]:
print(movies.head())

   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                              genres    year  \
0  [Adventure, Animation, Children, Comedy, Fantasy]  1995.0   
1                     [Adventure, Children, Fantasy]  1995.0   
2                                  [Comedy, Romance]  1995.0   
3                           [Comedy, Drama, Romance]  1995.0   
4                                           [Comedy]  1995.0   

                   clean_title  
0                    Toy Story  
1                      Jumanji  
2             Grumpier Old Men  
3            Waiting to Exhale  
4  Father of the Bride Part II  


In [68]:
#KNN modeli yaratmaq
model_knn = NearestNeighbors(metric='cosine', algorithm='brute')
model_knn.fit(user_item_matrix.values)

NearestNeighbors(algorithm='brute', metric='cosine')

In [69]:
#recommendaton algoritmi
from sklearn.neighbors import NearestNeighbors
import pandas as pd

user_item_matrix = ratings.pivot(index='userId', columns='movieId', values='rating').fillna(0)
print(user_item_matrix.shape)

# 1️⃣ KNN modeli yaratmaq
model_knn = NearestNeighbors(metric='cosine', algorithm='brute')
model_knn.fit(user_item_matrix.values)

# 2️⃣ Recommendation funksiyası
def recommend_movies(selected_movie_ids, user_item_matrix, movies, model_knn, n_recommendations=10):
    """
    selected_movie_ids: list of movieId selected by user
    user_item_matrix: pivoted user-item matrix
    movies: movies dataframe
    model_knn: trained NearestNeighbors model
    n_recommendations: number of recommended movies to return
    """

    # Yeni user vector
    temp_user = pd.Series(0, index=user_item_matrix.columns)
    for movie_id in selected_movie_ids:
        if movie_id in temp_user.index:
            temp_user[movie_id] = 5  # user rating 5 kimi qəbul edilir

    # Oxşar userləri tap
    distances, indices = model_knn.kneighbors(
        temp_user.values.reshape(1, -1),
        n_neighbors=6  # öz user + 5 oxşar user
    )

    # Tövsiyələri toplamaq
    recommendations = []
    for i in indices.flatten()[1:]:  # 0 → özü
        similar_user = user_item_matrix.iloc[i]
        unwatched = similar_user[temp_user == 0]
        high_rated = unwatched[unwatched >= 4]  # yalnız 4+ rating
        recommendations.extend(high_rated.index.tolist())

    # Duplicate-ları sil
    recommendations = list(set(recommendations))

    # Movie məlumatlarını qaytar
    recommended_movies = movies[
        movies['movieId'].isin(recommendations)
    ].head(n_recommendations)  # Top N

    return recommended_movies


(610, 9724)


In [70]:
#test edirik
# Məsələn user Toy Story (1), Jumanji (32) və Grumpier Old Men (260) seçdi
selected_movie_ids = [1, 32, 260]

recommended = recommend_movies(selected_movie_ids, user_item_matrix, movies, model_knn, n_recommendations=10)
print(recommended[['movieId','clean_title','genres','year']])


    movieId                  clean_title                         genres  \
4         5  Father of the Bride Part II                       [Comedy]   
5         6                         Heat      [Action, Crime, Thriller]   
8         9                 Sudden Death                       [Action]   
16       17        Sense and Sensibility               [Drama, Romance]   
24       25            Leaving Las Vegas               [Drama, Romance]   
55       62           Mr. Holland's Opus                        [Drama]   
58       65                     Bio-Dome                       [Comedy]   
66       74                 Bed of Roses               [Drama, Romance]   
84       95                 Broken Arrow  [Action, Adventure, Thriller]   
92      104                Happy Gilmore                       [Comedy]   

      year  
4   1995.0  
5   1995.0  
8   1995.0  
16  1995.0  
24  1995.0  
55  1995.0  
58  1996.0  
66  1996.0  
84  1996.0  
92  1996.0  


In [71]:
import requests

TMDB_API_KEY = "6488d38d72a69bcb051a50e054006e51"

def get_poster(title):
    try:
        url = f"https://api.themoviedb.org/3/search/movie?api_key={TMDB_API_KEY}&query={title}"
        response = requests.get(url)
        data = response.json()
        if data.get("results"):
            poster_path = data["results"][0].get("poster_path")
            if poster_path:
                return f"https://image.tmdb.org/t/p/w500{poster_path}"
    except:
        pass
    # fallback həmişə URL qaytarır
    return "https://via.placeholder.com/150x225?text=No+Poster"


In [72]:
import streamlit as st

st.title("🎬 Select the movies you love")

# Top 200 movie
top_movies = movies.head(100)
selected_movies = []

# Grid 5 kolona
for i in range(0, len(top_movies), 5):
    cols = st.columns(5)
    for j, (_, row) in enumerate(top_movies.iloc[i:i+5].iterrows()):
        col = cols[j]
        poster_url = get_poster(row['clean_title'])
        col.image(poster_url, use_column_width=True)
        if col.checkbox(row['clean_title'], key=row['movieId']):
            selected_movies.append(row['movieId'])

# Submit düyməsi
if st.button("Submit"):
    if selected_movies:
        recommended = recommend_movies(selected_movies, user_item_matrix, movies, model_knn, n_recommendations=20)
        st.subheader("🎯 Movies You May Love")
        for idx, row in recommended.iterrows():
            poster_url = get_poster(row['clean_title'])
            st.image(poster_url, width=150)
            st.caption(f"{row['clean_title']} ({row['year']})\nGenres: {', '.join(row['genres'])}")
    else:
        st.warning("Please select at least one movie!")


2026-02-18 22:13:58.476 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-18 22:13:58.478 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-18 22:13:58.479 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-18 22:13:58.482 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-18 22:13:58.484 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-18 22:13:58.485 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-18 22:13:58.486 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-18 22:13:58.487 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar